# Planners-5c — Le différentiel d'atteignabilité : ce que l'ajout d'une primitive rend possible

**Navigation** : [Index](../../README.md) | [<< Relaxation Lean (5b)](Planners-05b-Lean-Relaxation-Lean.ipynb) | [Domaines >>](Planners-06-Domains-Python.ipynb) | [Version Python](Planners-05c-Differentiel-Atteignabilite-Python.ipynb)

> « Les agents modifient leur vocabulaire » est une formule nébuleuse. Ce notebook la rend **testable**.

Le lake [`planning_lean`](../planning_lean/) — cité ici, jamais exécuté — prouve sans `sorry` que toute trajectoire réelle reste une trajectoire possible dans la **dynamique relaxée sans delete** (`run π s ⊆ runR π s`, `Planning/Relaxation.lean:56`, lemme clé `step ⊆ stepR` en `Planning/Strips.lean:58`), d'où l'admissibilité $h^+ \le h^*$ (`Planning/Admissibility.lean:41`). C'est un théorème sur **deux mondes issus du même vocabulaire** : la relaxation élargit ce que les deletes interdisaient.

La question de ce notebook est d'une autre nature :

> **quelle nouvelle classe de plans devient atteignable après l'ajout d'une primitive ?**

C'est un **différentiel d'atteignabilité** : $A_t \hookrightarrow A_{t+1}$, et l'on mesure ce que l'élargissement du vocabulaire rend possible que l'approfondissement de la recherche dans $A_t$ ne pouvait pas. Le protocole, en quatre pas :

1. **Un problème où aucune politique disponible n'atteint le but** — vérifié, pas supposé.
2. **L'agent cherche davantage dans le même $A_t$** — plus de budget, meilleure heuristique — et **échoue toujours**. Sans cette mesure de contrôle, on ne sait pas si l'extension a servi.
3. **L'agent est autorisé à inventer une primitive** — nommée, et **payante**.
4. **L'extension rend le but atteignable** — le delta est mesuré en buts, en coûts, en taille d'espace.

## Le moteur STRIPS, minimal et déterministe

Tout est en **BCL pure** : le sujet est le protocole de mesure, pas un solveur. Un état est un `HashSet<string>` de propositions, traité comme immuable et comparé par son contenu, un opérateur STRIPS un triplet (préconditions, ajouts, suppressions). Deux dynamiques cohabitent, exactement comme dans le lake :

- `Successeurs` — la dynamique **réelle** (`Step` : adds puis dels) ;
- `SuccesseursRelaxes` — la dynamique **sans delete** (`StepR` : adds seulement), la même construction que celle sur laquelle `step_subset_stepR` (`Planning/Strips.lean:58`) porte.

L'itération est triée partout : deux exécutions donnent les mêmes nombres (la mesure est déterministe, exigence de la série).

In [1]:
// Moteur STRIPS : step, stepR et parcours en largeur
using System;
using System.Collections.Generic;
using System.Globalization;
using System.Linq;

// Opérateur STRIPS : nom, préconditions, ajouts, suppressions (coût 1).
sealed class Operateur
{
    public string Nom { get; }
    public IReadOnlyCollection<string> Precond { get; }
    public IReadOnlyCollection<string> Ajouts { get; }
    public IReadOnlyCollection<string> Dels { get; }

    public Operateur(string nom, IEnumerable<string> precond, IEnumerable<string> ajouts, IEnumerable<string> dels)
    {
        Nom = nom;
        Precond = Array.AsReadOnly(precond.Distinct(StringComparer.Ordinal).ToArray());
        Ajouts = Array.AsReadOnly(ajouts.Distinct(StringComparer.Ordinal).ToArray());
        Dels = Array.AsReadOnly(dels.Distinct(StringComparer.Ordinal).ToArray());
    }

    public bool Applicable(HashSet<string> etat) => etat.IsSupersetOf(Precond);
    public override string ToString() => Nom;
}

// Une fois créé, un état n'est plus modifié ; les transitions travaillent sur une copie.
var comparateurEtats = HashSet<string>.CreateSetComparer();
HashSet<string> Etat(params string[] propositions) => new HashSet<string>(propositions, StringComparer.Ordinal);

HashSet<string> StepR(HashSet<string> etat, Operateur op)
{
    var t = new HashSet<string>(etat, StringComparer.Ordinal);
    t.UnionWith(op.Ajouts);
    return t;
}

HashSet<string> Step(HashSet<string> etat, Operateur op)
{
    var t = StepR(etat, op);
    t.ExceptWith(op.Dels);
    return t;
}

// Dynamique réelle (step) : tous les états successeurs, ordre déterministe.
IEnumerable<(Operateur Op, HashSet<string> Etat)> Successeurs(HashSet<string> etat, IEnumerable<Operateur> ops)
{
    foreach (var op in ops.OrderBy(o => o.Nom, StringComparer.Ordinal))
        if (op.Applicable(etat))
            yield return (op, Step(etat, op));
}

// Dynamique sans delete (stepR) : l'état ne fait que croître.
IEnumerable<(Operateur Op, HashSet<string> Etat)> SuccesseursRelaxes(HashSet<string> etat, IEnumerable<Operateur> ops)
{
    foreach (var op in ops.OrderBy(o => o.Nom, StringComparer.Ordinal))
        if (op.Applicable(etat) && !etat.IsSupersetOf(op.Ajouts))
            yield return (op, StepR(etat, op));
}

// Énumère exactement l'ensemble atteignable et conserve l'ordre de découverte pour départager les plans.
(Dictionary<HashSet<string>, int> Dist,
 Dictionary<HashSet<string>, (HashSet<string> Etat, Operateur Op)> Peres,
 int Expansions, List<HashSet<string>> OrdreDecouverte) Bfs(HashSet<string> initial, IReadOnlyList<Operateur> ops)
{
    var dist = new Dictionary<HashSet<string>, int>(comparateurEtats) { [initial] = 0 };
    var peres = new Dictionary<HashSet<string>, (HashSet<string> Etat, Operateur Op)>(comparateurEtats);
    var ordreDecouverte = new List<HashSet<string>> { initial };
    var file = new Queue<HashSet<string>>();
    file.Enqueue(initial);
    int expansions = 0;
    while (file.Count != 0)
    {
        var s = file.Dequeue();
        expansions++;
        foreach (var (op, t) in Successeurs(s, ops))
        {
            if (!dist.TryAdd(t, dist[s] + 1)) continue;
            peres.Add(t, (s, op));
            ordreDecouverte.Add(t);
            file.Enqueue(t);
        }
    }
    return (dist, peres, expansions, ordreDecouverte);
}

List<string> Reconstruire(IReadOnlyDictionary<HashSet<string>, (HashSet<string> Etat, Operateur Op)> peres,
                          HashSet<string> but)
{
    var plan = new List<string>();
    var s = but;
    while (peres.TryGetValue(s, out var pere))
    {
        plan.Add(pere.Op.Nom);
        s = pere.Etat;
    }
    plan.Reverse();
    return plan;
}

// BFS sans delete : la première apparition de chaque proposition donne sa distance relaxée.
Dictionary<string, int> DistancesRelaxees(HashSet<string> initial, IReadOnlyList<Operateur> ops)
{
    var distAtomes = initial.ToDictionary(p => p, p => 0, StringComparer.Ordinal);
    var vu = new HashSet<HashSet<string>>(comparateurEtats) { initial };
    var file = new Queue<(HashSet<string> Etat, int Distance)>();
    file.Enqueue((initial, 0));
    while (file.Count != 0)
    {
        var (s, d) = file.Dequeue();
        foreach (var (_, t) in SuccesseursRelaxes(s, ops))
        {
            if (!vu.Add(t)) continue;
            foreach (var p in t)
                if (!s.Contains(p)) distAtomes.TryAdd(p, d + 1);
            file.Enqueue((t, d + 1));
        }
    }
    return distAtomes;
}

// h_max : maximum des distances relaxées des atomes du but.
// Pour la chaîne h_max <= h+ <= h*, Admissibility.lean:41 est une référence citée.
double HMax(HashSet<string> etat, IReadOnlyCollection<string> but, IReadOnlyList<Operateur> ops)
{
    var dr = DistancesRelaxees(etat, ops);
    if (but.Any(p => !dr.ContainsKey(p))) return double.PositiveInfinity;
    return but.Max(p => dr[p]);
}

double CoutOptimal(IReadOnlyDictionary<HashSet<string>, int> dist, IReadOnlyCollection<string> but) =>
    dist.Where(kv => kv.Key.IsSupersetOf(but)).Select(kv => (double)kv.Value)
        .DefaultIfEmpty(double.PositiveInfinity).Min();

// Comparaison exacte, sans tolérance : les coûts sont des entiers ou l'infini.
// Le bilan final exploite les contrôles réellement exécutés dans chaque section.
var controlesParite = new List<(string Section, string Mesure, bool Conforme)>();
string Invariant(FormattableString texte) => texte.ToString(CultureInfo.InvariantCulture);
string AfficherCout(double cout, string texteInfini = "infini") =>
    double.IsPositiveInfinity(cout) ? texteInfini : cout.ToString(CultureInfo.InvariantCulture);
string ListePython(IEnumerable<string> elements) => "[" + string.Join(", ", elements.Select(e => "'" + e + "'")) + "]";

void Verifier<T>(string section, string mesure, T calcule, T attendu)
{
    bool conforme = EqualityComparer<T>.Default.Equals(calcule, attendu);
    controlesParite.Add((section, mesure, conforme));
    if (!conforme)
        throw new InvalidOperationException(Invariant($"{section} — {mesure} : calculé = {calcule} ; attendu = {attendu}."));
}

// Vérifier les deux transitions avant d'annoncer que le moteur est chargé.
var etatTemoin = Etat("avant");
var actionTemoin = new Operateur("transition-temoin", Etat("avant"), Etat("apres"), Etat("avant"));
var opsTemoin = new[] { actionTemoin };
Verifier("Moteur STRIPS", "step applique la suppression", Successeurs(etatTemoin, opsTemoin).Single().Etat.SetEquals(Etat("apres")), true);
Verifier("Moteur STRIPS", "stepR conserve le fluent", SuccesseursRelaxes(etatTemoin, opsTemoin).Single().Etat.SetEquals(Etat("avant", "apres")), true);
Console.WriteLine("Moteur STRIPS charge : step (reel) + stepR (relaxe sans delete).");

Moteur STRIPS charge : step (reel) + stepR (relaxe sans delete).


### Les deux dynamiques, face à face avec le lake

La correspondance avec le lake est terme à terme : là où `Planning/Strips.lean` définit un `Action F` (préconditions, ajouts, suppressions) et deux transitions — `step` (la réelle : ajouts **puis** suppressions) et `stepR` (la relaxée : ajouts **seulement**) — le moteur ci-dessus définit la même paire. Le lemme `step_subset_stepR` (`Strips.lean:58`) dit pourquoi la relaxée ne peut qu'élargir : appliquer un opérateur sans faire les suppressions rend l'état obtenu **sur-ensemble** de l'état réel — un état réel est toujours un état relaxé possible, jamais l'inverse.

Deux choix de mesure méritent d'être dits :

- **Déterminisme** : les opérateurs sont itérés triés par nom, les états sont des `HashSet<string>` traités comme immuables — deux exécutions produisent les mêmes nombres. Une mesure qui dérive d'une exécution à l'autre ne peut pas étalonner un différentiel.
- **Pourquoi `h_max` et pas `h⁺` direct** : le coût du plan relaxé **optimal** ($h^+$) est NP-difficile à calculer exactement ; $h_{max}$ — le maximum des distances relaxées par atome — se calcule exactement par BFS et minore $h^+$. La chaîne complète mesurable est donc $h_{max} \le h^+ \le h^*$ : le lake garantit le maillon droit (`Admissibility.lean:41`), le notebook mesure les deux extrémités calculables.

## Le domaine : deux rives, un gouffre, un radeau qui n'existe pas encore

Une grille $6 \times 3$ où la **colonne 3 est un gouffre** (aucune case) : rive gauche (colonnes 0-2), rive droite (colonnes 4-5). Le robot démarre en $(0,0)$. Sur la rive gauche : une **planchette** en $(1,1)$ et une **clé** en $(2,0)$. Sur la rive droite : un **trésor** en $(4,1)$.

Le vocabulaire disponible $A_t$ : **se déplacer** (4 directions, case cible existante) et **ramasser** (être sur la case de l'objet). Trois buts :

| But | Proposition | Rive |
|---|---|---|
| $g_1$ | atteindre la balise en $(5,2)$ | droite |
| $g_2$ | détenir le trésor | droite |
| $g_3$ | détenir la clé | **gauche** |

$g_3$ est le **témoin de santé** : s'il échoue, c'est le moteur qui est en cause, pas le gouffre.

In [2]:
// Construire la grille, les opérateurs de base et les buts
var colsGauche = new[] { 0, 1, 2 };
var colsDroite = new[] { 4, 5 };
var rangs = new[] { 0, 1, 2 };
var cases = colsGauche.Concat(colsDroite).SelectMany(c => rangs.Select(r => (C: c, R: r)))
    .Distinct().OrderBy(cr => cr.C).ThenBy(cr => cr.R).ToArray(); // colonne 3 = gouffre
var casesExistantes = cases.ToHashSet();

string Prop(int c, int r) => Invariant($"at-x{c}-y{r}");
var initial = Etat(Prop(0, 0), "plank-at-x1-y1", "key-at-x2-y0", "treasure-at-x4-y1");

List<Operateur> OpsDeBase()
{
    var ops = new List<Operateur>();
    var directions = new[] { (1, 0, "est"), (-1, 0, "ouest"), (0, 1, "nord"), (0, -1, "sud") };
    foreach (var (c, r) in cases)
    foreach (var (dc, dr, nomDir) in directions)
    {
        var cible = (c + dc, r + dr);
        if (casesExistantes.Contains(cible))
            ops.Add(new Operateur(Invariant($"move-{nomDir}-x{c}-y{r}"),
                                  Etat(Prop(c, r)), Etat(Prop(c + dc, r + dr)), Etat(Prop(c, r))));
    }

    foreach (var (objet, c, r) in new[] { ("plank", 1, 1), ("key", 2, 0), ("treasure", 4, 1) })
    {
        string auSol = Invariant($"{objet}-at-x{c}-y{r}");
        ops.Add(new Operateur($"pickup-{objet}", Etat(Prop(c, r), auSol),
                              Etat($"holding-{objet}"), Etat(auSol)));
    }
    return ops;
}

var aT = OpsDeBase();
const string g1 = "g1: balise (5,2)";
const string g2 = "g2: detenir le tresor";
const string g3 = "g3: detenir la cle";
var buts = new SortedDictionary<string, HashSet<string>>(StringComparer.Ordinal)
{
    [g1] = Etat(Prop(5, 2)),
    [g2] = Etat("holding-treasure"),
    [g3] = Etat("holding-key")
};

// Références citées : sortie de la cellule 5 et fluents du code source Python.
int nbMoves = aT.Count(o => o.Nom.StartsWith("move-", StringComparison.Ordinal));
int nbPickups = aT.Count(o => o.Nom.StartsWith("pickup-", StringComparison.Ordinal));
string libellesButs = ListePython(buts.Keys);
Verifier("Domaine", "cases existantes", cases.Length, 15);
Verifier("Domaine", "cases dans la colonne 3", cases.Count(cr => cr.C == 3), 0);
Verifier("Domaine", "opérateurs A_t", aT.Count, 41);
Verifier("Domaine", "moves", nbMoves, 38);
Verifier("Domaine", "pickups", nbPickups, 3);
Verifier("Domaine", "libellés ordonnés des buts", libellesButs,
    "['g1: balise (5,2)', 'g2: detenir le tresor', 'g3: detenir la cle']");
Verifier("Domaine", "fluents buts dans l'ordre", string.Join(" | ", buts.Values.Select(b => b.Single())),
    "at-x5-y2 | holding-treasure | holding-key");

Console.WriteLine(Invariant($"Cases existantes : {cases.Length} (colonne 3 = gouffre)"));
Console.WriteLine(Invariant($"Operateurs A_t   : {aT.Count} ({nbMoves} moves + {nbPickups} pickups)"));
Console.WriteLine($"Buts             : {libellesButs}");

Cases existantes : 15 (colonne 3 = gouffre)


Operateurs A_t   : 41 (38 moves + 3 pickups)


Buts             : ['g1: balise (5,2)', 'g2: detenir le tresor', 'g3: detenir la cle']


## Pas 1 — Vérifier l'inatteignabilité, pas la supposer

L'énumération exhaustive de l'ensemble atteignable est un **certificat fini** : si l'espace est énuméré en entier et que le but n'y figure pas, l'inatteignabilité est **exacte** — pas un délai dépassé, pas une impression. C'est l'exigence du protocole : le pas 1 ne dit pas « on n'a pas trouvé », il dit « **il n'y a pas** ».

In [3]:
// Énumérer les états atteignables dans A_t
// Références citées : verdicts et coûts affichés dans la cellule 7 du carnet Python.
var referenceBase = new Dictionary<string, (bool Atteignable, double Cout)>(StringComparer.Ordinal)
{
    [g1] = (false, double.PositiveInfinity),
    [g2] = (false, double.PositiveInfinity),
    [g3] = (true, 3.0)
};

var (distT, peresT, expT, ordreT) = Bfs(initial, aT);
var sAtteint = new HashSet<HashSet<string>>(distT.Keys, comparateurEtats);
Verifier("BFS de base", "états atteignables", sAtteint.Count, 36);
Verifier("BFS de base", "expansions", expT, 36);

Console.WriteLine(Invariant($"Ensemble atteignable dans A_t : {sAtteint.Count} etats, {expT} expansions (BFS exhaustif)"));
Console.WriteLine();
foreach (var (nom, but) in buts)
{
    bool ok = sAtteint.Any(s => s.IsSupersetOf(but));
    double hstar = CoutOptimal(distT, but);
    Verifier("BFS de base", $"{nom} — atteignabilité", ok, referenceBase[nom].Atteignable);
    Verifier("BFS de base", $"{nom} — h*", hstar, referenceBase[nom].Cout);
    Console.WriteLine(Invariant($"{nom,-28} atteignable={ok}   h*={AfficherCout(hstar)}"));
}

// Le certificat porte aussi sur l'absence des fluents de droite dans tout l'ensemble atteint.
var propositionsDroite = sAtteint.SelectMany(s => s)
    .Where(p => p.StartsWith("at-x4", StringComparison.Ordinal) ||
                p.StartsWith("at-x5", StringComparison.Ordinal) ||
                p.StartsWith("holding-treasure", StringComparison.Ordinal))
    .Distinct(StringComparer.Ordinal).OrderBy(p => p, StringComparer.Ordinal).ToArray();
string presenceDroite = propositionsDroite.Length == 0 ? "AUCUNE" : ListePython(propositionsDroite);
Verifier("BFS de base", "propositions de la rive droite", presenceDroite, "AUCUNE");
Console.WriteLine();
Console.WriteLine($"Propositions de la rive droite presentes dans l'ensemble atteignable : {presenceDroite}");

Ensemble atteignable dans A_t : 36 etats, 36 expansions (BFS exhaustif)


g1: balise (5,2)             atteignable=False   h*=infini


g2: detenir le tresor        atteignable=False   h*=infini


g3: detenir la cle           atteignable=True   h*=3


Propositions de la rive droite presentes dans l'ensemble atteignable : AUCUNE


### Lecture du pas 1 : le certificat et son architecture

L'ensemble atteignable vaut exactement **36 états** — et ce nombre n'est pas arbitraire : 9 cases de la rive gauche × 4 combinaisons d'inventaire (∅, planchette, clé, les deux). La rive droite apporte **zéro** état : aucune proposition `at-x4-*`/`at-x5-*`/`holding-treasure` n'apparaît jamais. Le BFS a énuméré l'espace **en entier** (36 expansions pour 36 états : chaque état a été dépilé exactement une fois) — l'inatteignabilité de $g_1$ et $g_2$ est donc un **certificat fini**, à distinguer soigneusement d'un échec de recherche : un solveur qui « n'a pas trouvé en 10 secondes » ne prouve rien ; une énumération complète qui ne contient pas le but prouve l'impossibilité.

Le témoin de santé fait son office : $g_3$ est atteignable avec $h^* = 3$ (deux déplacements jusqu'à la clé en $(2,0)$, un pickup). Le moteur marche, le gouffre seul est en cause.

## Pas 2 — La mesure de contrôle : chercher *davantage* dans le même $A_t$

C'est le pas que l'on a le plus envie de sauter, et le grain entier s'effondre sans lui. Trois niveaux d'effort, **même vocabulaire** :

1. **BFS exhaustif** — l'exploration complète (déjà faite, budget : 36 états) ;
2. **A\*** avec l'heuristique admissible $h_{max}$ — la meilleure heuristique calculable issue de la relaxation du lake (chaîne $h_{max} \le h^+ \le h^*$) ;
3. **IDDFS** (approfondissement itératif, profondeur 24) — l'effort systématique sans mémoire.

Si le but devient atteignable à un seul de ces niveaux, l'extension du pas 3 n'aurait rien mesuré : c'était un problème de recherche, pas de vocabulaire.

In [4]:
// Comparer BFS, A* avec h_max et IDDFS
// A* : priorité (f, g, ordre d'insertion), avec marquage à l'insertion comme dans la source.
(bool Atteint, int Expansions, int? Cout, HashSet<HashSet<string>> EtatsDepiles) AStarHMax(
    HashSet<string> depart, HashSet<string> but, IReadOnlyList<Operateur> ops)
{
    var frontiere = new PriorityQueue<HashSet<string>, (double F, int G, long Ordre)>();
    var vus = new HashSet<HashSet<string>>(comparateurEtats) { depart };
    var depiles = new HashSet<HashSet<string>>(comparateurEtats);
    long ordre = 0;
    int expansions = 0;
    frontiere.Enqueue(depart, (HMax(depart, but, ops), 0, ordre++));
    while (frontiere.TryDequeue(out var s, out var priorite))
    {
        expansions++;
        depiles.Add(s);
        if (s.IsSupersetOf(but)) return (true, expansions, priorite.G, depiles);
        foreach (var (_, t) in Successeurs(s, ops))
        {
            if (!vus.Add(t)) continue;
            int g = priorite.G + 1;
            // Une priorité infinie n'élimine pas un état de la frontière.
            frontiere.Enqueue(t, (g + HMax(t, but, ops), g, ordre++));
        }
    }
    return (false, expansions, null, depiles);
}

// IDDFS : une pile LIFO par seuil, sans réutiliser le marquage entre deux seuils.
(bool Atteint, int Expansions, int? Cout, HashSet<HashSet<string>> EtatsDepiles) Iddfs(
    HashSet<string> depart, HashSet<string> but, IReadOnlyList<Operateur> ops, int profondeurMax)
{
    int expansions = 0;
    // Ce relevé transversal sert uniquement à comparer les espaces explorés, jamais à filtrer la pile.
    var depiles = new HashSet<HashSet<string>>(comparateurEtats);
    for (int limite = 0; limite <= profondeurMax; limite++)
    {
        var pile = new Stack<(HashSet<string> Etat, int Profondeur)>();
        pile.Push((depart, 0));
        // Un ensemble par profondeur réalise le marquage des couples (état, profondeur).
        var vusParProfondeur = Enumerable.Range(0, limite + 1)
            .Select(_ => new HashSet<HashSet<string>>(comparateurEtats)).ToArray();
        while (pile.Count != 0)
        {
            var (s, d) = pile.Pop();
            expansions++;
            depiles.Add(s);
            if (s.IsSupersetOf(but)) return (true, expansions, d, depiles);
            if (d >= limite) continue;
            foreach (var (_, t) in Successeurs(s, ops))
                if (vusParProfondeur[d + 1].Add(t)) pile.Push((t, d + 1));
        }
    }
    return (false, expansions, null, depiles);
}

// Référence citée : tableau de la cellule 10 du carnet Python.
const int profondeurControle = 24;
var referenceEfforts = new Dictionary<string, (bool Bfs, int ExpBfs, bool AStar, int ExpAStar, bool Iddfs, int ExpIddfs)>(StringComparer.Ordinal)
{
    [g1] = (false, 36, false, 36, false, 4151),
    [g2] = (false, 36, false, 36, false, 4151),
    [g3] = (true, 36, true, 4, true, 24)
};
var rapport = new List<(string Nom, bool Bfs, int ExpBfs, bool AStar, int ExpAStar, bool Iddfs, int ExpIddfs)>();
foreach (var (nom, but) in buts)
{
    bool okBfs = sAtteint.Any(s => s.IsSupersetOf(but));
    var a = AStarHMax(initial, but, aT);
    var i = Iddfs(initial, but, aT, profondeurControle);
    var mesure = (Bfs: okBfs, ExpBfs: expT, AStar: a.Atteint, ExpAStar: a.Expansions, Iddfs: i.Atteint, ExpIddfs: i.Expansions);
    Verifier("Efforts de recherche", $"{nom} — succès et expansions BFS / A* / IDDFS", mesure, referenceEfforts[nom]);
    if (nom == g1 || nom == g2)
    {
        Verifier("Efforts de recherche", $"{nom} — mêmes états dépilés en BFS et A*", a.EtatsDepiles.SetEquals(sAtteint), true);
        Verifier("Efforts de recherche", $"{nom} — mêmes états dépilés en BFS et IDDFS", i.EtatsDepiles.SetEquals(sAtteint), true);
    }
    rapport.Add((nom, mesure.Bfs, mesure.ExpBfs, mesure.AStar, mesure.ExpAStar, mesure.Iddfs, mesure.ExpIddfs));
}

var echecsCommuns = rapport.Where(r => !r.Bfs && !r.AStar && !r.Iddfs).Select(r => r.Nom).ToArray();
Verifier("Efforts de recherche", "buts en échec aux trois niveaux", ListePython(echecsCommuns), ListePython(new[] { g1, g2 }));
string VerdictEffort(bool ok) => ok ? "OK" : "ECHEC";
string titreIddfs = Invariant($"IDDFS({profondeurControle})");
Console.WriteLine(Invariant($"{"but",-28} {"BFS",10} {"A*+h_max",18} {titreIddfs,18}"));
foreach (var (nom, b, eb, a, ea, i, ei) in rapport)
    Console.WriteLine(Invariant($"{nom,-28} {VerdictEffort(b),7}({eb,3}) {VerdictEffort(a),8}({ea,4}) {VerdictEffort(i),8}({ei,5})"));

string premiersButs = string.Join(" et ", echecsCommuns.Select(nom => nom.Split(':')[0]));
Console.WriteLine();
Console.WriteLine($"Verdict du controle : {premiersButs} echouent aux TROIS niveaux d'effort.");
Console.WriteLine(Invariant($"L'espace atteignable est le MEME ({sAtteint.Count} etats) quel que soit le budget."));

but                                 BFS           A*+h_max          IDDFS(24)


g1: balise (5,2)               ECHEC( 36)    ECHEC(  36)    ECHEC( 4151)


g2: detenir le tresor          ECHEC( 36)    ECHEC(  36)    ECHEC( 4151)


g3: detenir la cle                OK( 36)       OK(   4)       OK(   24)


Verdict du controle : g1 et g2 echouent aux TROIS niveaux d'effort.


L'espace atteignable est le MEME (36 etats) quel que soit le budget.


### Lecture du pas 2 : l'asymétrie des budgets est la signature

Le tableau se lit en deux colonnes morales. Sur le but **atteignable** $g_3$, l'effort paie de façon spectaculaire : BFS énumère les 36 états, A\* avec $h_{max}$ n'en développe que **4** (l'heuristique admissible le mène droit au but), IDDFS en re-développe 24. C'est le régime ordinaire de la planification : *plus d'effort, une meilleure heuristique — le même problème devient tractable*.

Sur les buts **inatteignables**, l'effet disparaît : BFS et A\* rendent le même verdict en énumérant **le même espace de 36 états** — la meilleure heuristique du monde ne peut pas indiquer un chemin qui n'existe pas, elle ne peut qu'épuiser. IDDFS, lui, **brûle 4 151 expansions** — plus de cent fois la taille de l'espace, parce qu'il ré-explorre sans mémoire à chaque seuil de profondeur — pour conclure la même chose. C'est l'asymétrie qui authentifie le diagnostic :

- budgets qui **s'améliorent** sur un but → le problème était de *recherche* ;
- budgets qui **brûlent sans rien rapporter** → le problème est de *vocabulaire*.

Et le résultat négatif de la relaxation (section suivante) complète le verdict : même le monde relaxé — pourtant plus grand par construction — ne contient pas la rive droite. Le trou n'est pas dans les deletes, il est dans l'alphabet des actions.

## Ce que le lake garantit — et pourquoi la relaxation ne peut pas sauver $g_1$

Le théorème d'admissibilité (`Planning/Admissibility.lean:41`) dit : tout plan réel est un plan relaxé, donc $h^+ \le h^*$. La relaxation **sans delete** (`run ⊆ runR`, `Planning/Relaxation.lean:56`) est un élargissement de monde : elle rend atteignable tout ce que les suppressions d'effets interdisaient. Mesurons-la ici :

- sur la rive **gauche**, la relaxation minorer honnêtement : $h_{max} \le h^*$, parfois **strictement** — le but conjonctif « planchette **et** clé **et** revenir au départ » coûte 8 en réel mais paraît plus court au monde relaxé (les positions s'accumulent, les détours disparaissent) ;
- sur $g_1$/$g_2$, la distance relaxée est **infinie elle aussi** : aucun opérateur de $A_t$, même relaxé, ne produit une proposition de la rive droite. La relaxation élargit les mondes **par les deletes** ; une **primitive manquante** est un trou que même le monde relaxé ne comble pas.

In [5]:
// Mesurer la borne relaxée sur le but conjonctif et les buts de droite
// (a) La relaxation minore, parfois strictement, sur la rive gauche
// Références citées : coûts et verdict de la cellule 13 du carnet Python.
var butConjonctif = Etat("holding-plank", "holding-key", Prop(0, 0));
double hstarConj = CoutOptimal(distT, butConjonctif);
double hmaxConj = HMax(initial, butConjonctif, aT);
bool borneConjValide = hmaxConj <= hstarConj;
string verdictConj = !borneConjValide ? "VIOLEE" : hmaxConj < hstarConj ? "STRICT" : "egalite";
Verifier("Relaxation de base", "but conjonctif — h*", hstarConj, 8.0);
Verifier("Relaxation de base", "but conjonctif — h_max", hmaxConj, 3.0);
Verifier("Relaxation de base", "but conjonctif — h_max <= h*", borneConjValide, true);
Verifier("Relaxation de base", "but conjonctif — verdict", verdictConj, "STRICT");
Console.WriteLine("But conjonctif (planchette ET cle ET revenir au depart)");
Console.WriteLine(Invariant($"  h* (reel, BFS uniforme)  = {hstarConj}"));
Console.WriteLine(Invariant($"  h_max (relaxe, exact)    = {hmaxConj}"));
Console.WriteLine(Invariant($"  chaine verifiee          : h_max ({hmaxConj}) <= h* ({hstarConj})  -> {verdictConj}"));
// Référence citée : cette ligne reprend le rappel théorique du Python, pas une preuve exécutée ici.
Console.WriteLine("  (le lake garantit h+ <= h* en Admissibility.lean:41 ; h_max <= h+ complete la chaine)");

// (b) Sur g1/g2 : meme le monde relaxe est infini
foreach (var nom in new[] { g1, g2 })
{
    double hm = HMax(initial, buts[nom], aT);
    Verifier("Relaxation de base", $"{nom} — h_max", hm, double.PositiveInfinity);
    Console.WriteLine();
    Console.WriteLine($"{nom}: h_max dans A_t = {AfficherCout(hm, "INFINI (relaxement inatteignable)")}");
}

But conjonctif (planchette ET cle ET revenir au depart)


  h* (reel, BFS uniforme)  = 8


  h_max (relaxe, exact)    = 3


  chaine verifiee          : h_max (3) <= h* (8)  -> STRICT


  (le lake garantit h+ <= h* en Admissibility.lean:41 ; h_max <= h+ complete la chaine)


g1: balise (5,2): h_max dans A_t = INFINI (relaxement inatteignable)


g2: detenir le tresor: h_max dans A_t = INFINI (relaxement inatteignable)


## Pas 3 — La primitive nommée, et ce qu'elle coûte

Une extension **gratuite** ne mesure rien : si la nouvelle action ne coûte rien, le delta d'atteignabilité n'a pas de contrefactuel. La primitive inventée ici est le **radeau** :

> `raft-across-y{r}` — préconditions : être en $(2,r)$ **et** détenir la planchette. Effets : atterrir en $(4,r)$ ; **la planchette est consommée** (suppression `holding-plank`).

Son coût est double et visible dans les plans du pas 4 : (1) la planchette est **dépensée** — un seul trajet, pas d'aller-retour gratuit ; (2) l'aller chercher en $(1,1)$ est un **détour** imposé avant la traversée. Le vocabulaire s'élargit : $A_t \hookrightarrow A_{t+1}$.

In [6]:
// Étendre le vocabulaire avec les trois actions du radeau
List<Operateur> OpsAvecRadeau()
{
    var ops = OpsDeBase();
    // Une traversée par rangée : quitter la rive gauche et consommer la planchette.
    foreach (int r in rangs)
    {
        var departAvecPlanchette = Etat(Prop(2, r), "holding-plank");
        ops.Add(new Operateur(Invariant($"raft-across-y{r}"), departAvecPlanchette,
                              Etat(Prop(4, r)), departAvecPlanchette));
    }
    return ops;
}

// Référence citée : effectifs et signatures affichés dans la cellule 15 du carnet Python.
var referenceRadeau = new Dictionary<string, (string Precond, string Ajouts, string Dels)>(StringComparer.Ordinal)
{
    ["raft-across-y0"] = ("['at-x2-y0', 'holding-plank']", "['at-x4-y0']", "['at-x2-y0', 'holding-plank']"),
    ["raft-across-y1"] = ("['at-x2-y1', 'holding-plank']", "['at-x4-y1']", "['at-x2-y1', 'holding-plank']"),
    ["raft-across-y2"] = ("['at-x2-y2', 'holding-plank']", "['at-x4-y2']", "['at-x2-y2', 'holding-plank']")
};
var aT1 = OpsAvecRadeau();
var nouveaux = aT1.Where(o => o.Nom.StartsWith("raft-", StringComparison.Ordinal))
    .OrderBy(o => o.Nom, StringComparer.Ordinal).ToArray();
string nomsRadeau = ListePython(nouveaux.Select(o => o.Nom));
Verifier("Vocabulaire étendu", "opérateurs A_t", aT.Count, 41);
Verifier("Vocabulaire étendu", "opérateurs A_t+1", aT1.Count, 44);
Verifier("Vocabulaire étendu", "noms ordonnés des actions du radeau", nomsRadeau,
    ListePython(referenceRadeau.Keys.OrderBy(nom => nom, StringComparer.Ordinal)));

string ListeTriee(IEnumerable<string> fluents) => ListePython(fluents.OrderBy(p => p, StringComparer.Ordinal));
Console.WriteLine(Invariant($"A_t   : {aT.Count} operateurs"));
Console.WriteLine(Invariant($"A_t+1 : {aT1.Count} operateurs (primitive ajoutee : {nomsRadeau})"));
foreach (var op in nouveaux)
{
    string precond = ListeTriee(op.Precond);
    string ajouts = ListeTriee(op.Ajouts);
    string dels = ListeTriee(op.Dels);
    Verifier("Vocabulaire étendu", $"{op.Nom} — préconditions, ajouts et suppressions",
        (precond, ajouts, dels), referenceRadeau[op.Nom]);
    Console.WriteLine($"  {op.Nom}: precond={precond} ajouts={ajouts} supprime={dels}");
}

A_t   : 41 operateurs


A_t+1 : 44 operateurs (primitive ajoutee : ['raft-across-y0', 'raft-across-y1', 'raft-across-y2'])


  raft-across-y0: precond=['at-x2-y0', 'holding-plank'] ajouts=['at-x4-y0'] supprime=['at-x2-y0', 'holding-plank']


  raft-across-y1: precond=['at-x2-y1', 'holding-plank'] ajouts=['at-x4-y1'] supprime=['at-x2-y1', 'holding-plank']


  raft-across-y2: precond=['at-x2-y2', 'holding-plank'] ajouts=['at-x4-y2'] supprime=['at-x2-y2', 'holding-plank']


## Pas 4 — Le delta, mesuré

Trois mesures, pas une impression :

1. **En buts** : quels éléments de $G$ deviennent atteignables — le différentiel $\Delta = \text{atteignable}(A_{t+1}) \setminus \text{atteignable}(A_t)$, but par but ;
2. **En plans** : le plan pour $g_1$ et celui pour $g_2$, exhibés action par action avec leur coût $h^*$ ;
3. **En espace** : la taille de l'ensemble atteignable — le monde lui-même a grandi, ce n'est pas la recherche qui a approfondi.

In [7]:
// Mesurer le delta et reconstruire les deux plans optimaux
// Références citées : effectifs, verdicts, coûts et plans affichés dans la cellule 17 du carnet Python.
var referenceDelta = new Dictionary<string, (bool Avant, bool Apres, double Cout)>(StringComparer.Ordinal)
{
    [g1] = (false, true, 7.0),
    [g2] = (false, true, 6.0),
    [g3] = (true, true, 3.0)
};
var referencePlans = new Dictionary<string, string[]>(StringComparer.Ordinal)
{
    [g1] = new[] { "move-est-x0-y0", "move-nord-x1-y0", "pickup-plank", "move-est-x1-y1", "move-nord-x2-y1", "raft-across-y2", "move-est-x4-y2" },
    [g2] = new[] { "move-est-x0-y0", "move-nord-x1-y0", "pickup-plank", "move-est-x1-y1", "raft-across-y1", "pickup-treasure" }
};

var (distT1, peresT1, expT1, ordreT1) = Bfs(initial, aT1);
var sAtteint1 = new HashSet<HashSet<string>>(distT1.Keys, comparateurEtats);
int deltaEtats = sAtteint1.Count - sAtteint.Count;
Verifier("Différentiel", "états avant", sAtteint.Count, 36);
Verifier("Différentiel", "états après", sAtteint1.Count, 60);
Verifier("Différentiel", "chaque état après extension est dépilé une fois", expT1, sAtteint1.Count);
Verifier("Différentiel", "inclusion de l'espace de base", sAtteint.IsSubsetOf(sAtteint1), true);
Verifier("Différentiel", "nouveaux états", sAtteint1.Count(s => !sAtteint.Contains(s)), 24);
Verifier("Différentiel", "écart de taille", deltaEtats, 24);

var nouveauxButs = new List<string>();
Console.WriteLine(Invariant($"Ensemble atteignable : |A_t| = {sAtteint.Count} etats  ->  |A_t+1| = {sAtteint1.Count} etats (+{deltaEtats})"));
Console.WriteLine();
Console.WriteLine(Invariant($"{"but",-28} {"A_t",8} {"A_t+1",8} {"h*(A_t+1)",10}  delta"));
foreach (var (nom, but) in buts)
{
    bool avant = sAtteint.Any(s => s.IsSupersetOf(but));
    bool apres = sAtteint1.Any(s => s.IsSupersetOf(but));
    double h1 = CoutOptimal(distT1, but);
    Verifier("Différentiel", $"{nom} — verdicts avant/après et h* après", (avant, apres, h1), referenceDelta[nom]);
    bool nouveau = apres && !avant;
    if (nouveau) nouveauxButs.Add(nom);
    string etiquetteDelta = nouveau ? "NOUVELLEMENT ATTEIGNABLE" : "";
    Console.WriteLine(Invariant($"{nom,-28} {VerdictEffort(avant),8} {VerdictEffort(apres),8} {AfficherCout(h1),10}  {etiquetteDelta}"));
}
Verifier("Différentiel", "ensemble exact des nouveaux buts", ListePython(nouveauxButs), ListePython(new[] { g1, g2 }));
Verifier("Différentiel", "g3 — coût inchangé", CoutOptimal(distT1, buts[g3]), CoutOptimal(distT, buts[g3]));

// Le premier état but découvert par le BFS a un coût minimal ; cet ordre départage les ex æquo comme en Python.
foreach (var (nom, titrePlan) in new[] { (g1, "g1 (balise"), (g2, "g2 (tresor") })
{
    var etatBut = ordreT1.First(s => s.IsSupersetOf(buts[nom]));
    int coutPlan = distT1[etatBut];
    var plan = Reconstruire(peresT1, etatBut);
    Verifier("Différentiel", $"{nom} — coût du plan", (double)coutPlan, referenceDelta[nom].Cout);
    Verifier("Différentiel", $"{nom} — longueur du plan", plan.Count, coutPlan);
    Verifier("Différentiel", $"{nom} — séquence exacte", ListePython(plan), ListePython(referencePlans[nom]));

    // Rejouer le plan pour vérifier ses préconditions et son état final.
    var etatRejoue = initial;
    foreach (string action in plan)
    {
        var op = aT1.Single(o => o.Nom == action);
        Verifier("Différentiel", $"{nom} — {action} applicable", op.Applicable(etatRejoue), true);
        etatRejoue = Step(etatRejoue, op);
    }
    Verifier("Différentiel", $"{nom} — état final du plan", etatRejoue.SetEquals(etatBut), true);

    Console.WriteLine();
    Console.WriteLine(Invariant($"Plan {titrePlan}, h* = {coutPlan}) :"));
    for (int i = 0; i < plan.Count; i++)
        Console.WriteLine(Invariant($"  {i + 1,2}. {plan[i]}"));
}

Ensemble atteignable : |A_t| = 36 etats  ->  |A_t+1| = 60 etats (+24)


but                               A_t    A_t+1  h*(A_t+1)  delta


g1: balise (5,2)                ECHEC       OK          7  NOUVELLEMENT ATTEIGNABLE


g2: detenir le tresor           ECHEC       OK          6  NOUVELLEMENT ATTEIGNABLE


g3: detenir la cle                 OK       OK          3  


Plan g1 (balise, h* = 7) :


... (10 outputs left out)

   3. pickup-plank


   4. move-est-x1-y1


   5. raft-across-y1


   6. pickup-treasure


### Lecture du delta : ce que les trois mesures disent ensemble

- **$\Delta = \{g_1, g_2\}$** — les deux buts de la rive droite deviennent atteignables ; $g_3$, le témoin de santé, était déjà atteignable et le reste : l'extension n'a rien cassé.
- **Les plans racontent le coût de la primitive** : le plan de $g_1$ commence par le détour $(0,0) \to (1,0) \to (1,1)$ pour la planchette, et la traversée **consomme** la planchette — si le but exigeait un retour, il faudrait réinventer. Le plan de $g_2$ suit le même squelette et s'arrête une case plus tôt : la traversée en **rangée 1** atterrit pile sur le trésor $(4,1)$ — un pickup de plus, deux déplacements de moins, d'où $h^*(g_2) = 6 < h^*(g_1) = 7$.
- **L'espace a grandi, pas la recherche** : l'ensemble atteignable passe de **36 à 60 états (+24)** — des états de la rive droite **existent maintenant**, alors que les trois niveaux d'effort du pas 2 énuméraient inlassablement les mêmes 36. C'est la différence entre approfondir $A_t$ et élargir en $A_{t+1}$ : le différentiel de Laborit, mesuré.

Et la relaxation ? Dans $A_{t+1}$, $h_{max}(g_1)$ devient fini — le monde relaxé suit le vocabulaire, jamais l'inverse.

In [8]:
// Vérifier les bornes après extension et attester la parité
// Référence citée : bornes, coûts et verdicts affichés dans la cellule 19 du carnet Python.
var referenceBornes = new Dictionary<string, (double Avant, double Apres, double Cout, string Verdict)>(StringComparer.Ordinal)
{
    [g1] = (double.PositiveInfinity, 7.0, 7.0, "OK"),
    [g2] = (double.PositiveInfinity, 6.0, 6.0, "OK")
};
foreach (var nom in new[] { g1, g2 })
{
    var but = buts[nom];
    double hmAvant = HMax(initial, but, aT);
    double hmApres = HMax(initial, but, aT1);
    double hstar1 = CoutOptimal(distT1, but);
    var attendu = referenceBornes[nom];
    string verdictBorne = hmApres <= hstar1 ? "OK" : "VIOLEE";
    Verifier("Relaxation après extension", $"{nom} — h_max avant", hmAvant, attendu.Avant);
    Verifier("Relaxation après extension", $"{nom} — h_max après", hmApres, attendu.Apres);
    Verifier("Relaxation après extension", $"{nom} — h* après", hstar1, attendu.Cout);
    Verifier("Relaxation après extension", $"{nom} — chaîne h_max<=h*", verdictBorne, attendu.Verdict);
    Console.WriteLine(Invariant($"{nom,-28} h_max: A_t={AfficherCout(hmAvant, "inf")} -> A_t+1={AfficherCout(hmApres)}   h*(A_t+1)={AfficherCout(hstar1)}   chaine h_max<=h*: {verdictBorne}"));
}

// Bilan des comparaisons exactes réellement exécutées, sans accepter de section vide.
var sectionsParite = new[] { "Moteur STRIPS", "Domaine", "BFS de base", "Efforts de recherche", "Relaxation de base", "Vocabulaire étendu", "Différentiel", "Relaxation après extension" };
var bilanParite = controlesParite.ToLookup(c => c.Section, StringComparer.Ordinal);
Console.WriteLine();
Console.WriteLine("Attestation de parité — comparaisons exactes avec les références Python et contrôles des invariants :");
foreach (string section in sectionsParite)
{
    int effectif = bilanParite[section].Count();
    if (effectif == 0)
        throw new InvalidOperationException($"Section sans contrôle de parité : {section}.");
    int ecarts = bilanParite[section].Count(c => !c.Conforme);
    string statut = ecarts == 0 ? "CONFORME" : "ÉCART";
    Console.WriteLine(Invariant($"  {section} : {effectif} contrôles, {ecarts} écarts — {statut}"));
}

g1: balise (5,2)             h_max: A_t=inf -> A_t+1=7   h*(A_t+1)=7   chaine h_max<=h*: OK


g2: detenir le tresor        h_max: A_t=inf -> A_t+1=6   h*(A_t+1)=6   chaine h_max<=h*: OK


Attestation de parité — comparaisons exactes avec les références Python et contrôles des invariants :


  Moteur STRIPS : 2 contrôles, 0 écarts — CONFORME


  Domaine : 7 contrôles, 0 écarts — CONFORME


  BFS de base : 9 contrôles, 0 écarts — CONFORME


  Efforts de recherche : 8 contrôles, 0 écarts — CONFORME


  Relaxation de base : 6 contrôles, 0 écarts — CONFORME


  Vocabulaire étendu : 6 contrôles, 0 écarts — CONFORME


  Différentiel : 32 contrôles, 0 écarts — CONFORME


  Relaxation après extension : 8 contrôles, 0 écarts — CONFORME


## Résumé — le test opérationnel de la strate 7

| Pas | Mesure | Résultat |
|---|---|---|
| 1. Inatteignabilité vérifiée | énumération exhaustive | 36 états, $g_1, g_2$ absents — certificat exact |
| 2. Contrôle d'effort | BFS / A\*+$h_{max}$ / IDDFS(24) | les trois échouent, **même espace** : c'est un problème de vocabulaire |
| 3. Primitive nommée | `raft-across-y{r}` | coût : planchette consommée + détour $(1,1)$ |
| 4. Delta | buts, plans, espace | $\Delta = \{g_1, g_2\}$, plans exhibés ($h^*$ 7 et 6), l'espace atteignable 36 → 60 |

**Ce que le certificat couvre et ne couvre pas.** Le lake garantit les énoncés *structurels* : $step \subseteq stepR$ (`Strips.lean:58`), $run \subseteq runR$ (`Relaxation.lean:56`), $h^+ \le h^*$ (`Admissibility.lean:41`). Il ne dit **rien** du choix d'une primitive, ni de l'optimisation du chercheur : ce qui est certifié est la *garantie de l'heuristique*, ce qui est mesuré ici est *l'instance*. Les autres sémantiques (préférences, temporalité, incertitude) restent hors de ce certificat comme de cette mesure.

**Le protocole, réutilisable.** Toute revendication du type « l'agent a appris une nouvelle capacité » passe le même banc : (1) but inatteignable prouvé, (2) l'effort accru échoue, (3) la primitive est nommée et payante, (4) le delta est un ensemble ou un chiffre. Sans le pas 2, chercher davantage et élargir sont indiscernables ; sans le pas 3, le delta ne mesure rien.

## Limites et suites

Ce que ce banc ne fait **pas**, pour être lu honnêtement :

- **Ce n'est pas une étude de solveur**. Fast Downward et OR-Tools CP-SAT ([Planners-4](Planners-04-Fast-Downward-CSharp.ipynb), [Planners-7](../03-Advanced/Planners-07-OR-Tools-Python.ipynb)) traitent des espaces que ce mini-moteur ne peut pas énumérer ; ici, l'énumération exhaustive est précisément l'**instrument de mesure** — c'est elle qui transforme « on n'a pas trouvé » en certificat. Sur un espace exponentiel, le pas 1 devrait remplacer l'énumération par une preuve d'inatteignabilité (invariants, abstraction) — le protocole reste, l'instrument change.
- **$h^+$ exact n'est pas calculé** — NP-difficile en général ; la chaîne mesurée s'arrête à ses bornes calculables ($h_{max}$, $h^*$). Le companion formel [Planners-5b](Planners-05b-Lean-Relaxation-Lean.ipynb) exécute le lemme d'admissibilité dans le kernel Lean lui-même.
- **La synthèse reste hors Lean** : ce qui est certifié est la *garantie structurelle* de la relaxation, pas le choix de la primitive. Un `raft-across` mal fichu (qui traverserait sans planchette) passerait le même banc — l'exercice 2 montre le versant symétrique : une extension stérile y échoue.

**Suite naturelle** : le différentiel se généralise en remove-differential (quelle classe de plans **disparaît** quand on retire une primitive) et en coût-differential (même vocabulaire, coût des actions changé) — le même protocole à quatre pas s'applique, seul le pas 3 change de geste.

## Exercices

### Exercice 1 — Une primitive *gratuite* : le deltaplane

Ajoutez l'opérateur `deltaplane-y{r}` : précondition être en $(0,r)$ (bord **gauche**), effet atterrir en $(4,r)$, **sans planchette**. Recalculez $\Delta$ et $h^*(g_1)$ avec ce vocabulaire $A_{t+1}'$. Le delta change-t-il ? Le coût, oui ou non, et pourquoi le pas 3 du protocole exige-t-il de le dire ?

In [9]:
// Exercice 1 a completer : construire aT1Prime avec deltaplane-y{r}, recalculer
// le delta de buts et h*(g1), puis comparer au radeau.
// aT1Prime = ...
// resultExo1 : (delta, hstarG1)
(IReadOnlyCollection<string> Delta, double HstarG1)? resultExo1 = null; // TODO etudiant
Console.WriteLine("Exercice 1 a completer : delta et h*(g1) avec le deltaplane.");

Exercice 1 a completer : delta et h*(g1) avec le deltaplane.


### Exercice 2 — Une extension *stérile* : jeter la planchette

Ajoutez l'opérateur `discard-plank` : précondition `holding-plank`, effet suppression de `holding-plank` (rien d'ajouté). Le vocabulaire s'élargit pourtant — mesurez $\Delta$ et la taille de l'ensemble atteignable. Que démontre ce contre-exemple sur le protocole ?

In [10]:
// Exercice 2 a completer : construire aT1Sterile = aT + discard-plank,
// mesurer Delta et comparer la taille de l'espace atteignable.
// resultExo2 : (delta, tailleAtteignable)
(IReadOnlyCollection<string> Delta, int TailleAtteignable)? resultExo2 = null; // TODO etudiant
Console.WriteLine("Exercice 2 a completer : delta et taille de l'espace avec discard-plank.");

Exercice 2 a completer : delta et taille de l'espace avec discard-plank.


### Exercice 3 — La chaîne $h_{max} \le h^+ \le h^*$ sur $g_3$

Sur le but $g_3$ (détenir la clé), calculez $h_{max}$ et $h^*$ dans $A_t$. Sont-ils égaux ? Justifiez votre verdict en une phrase et comparez-le au cas du but conjonctif de la rive gauche ($h_{max} = 3$ contre $h^* = 8$).

In [11]:
// Exercice 3 a completer : h_max(g3) et h*(g3) dans A_t, verdict egalite/strict,
// et la phrase d'explication dans explicationExo3.
// resultExo3 : (hmaxG3, hstarG3)
// explicationExo3 : une phrase de justification.
(double HmaxG3, double HstarG3)? resultExo3 = null; // TODO etudiant
string explicationExo3 = ""; // TODO etudiant
Console.WriteLine("Exercice 3 a completer : h_max(g3), h*(g3), explication.");

Exercice 3 a completer : h_max(g3), h*(g3), explication.


## À retenir

| Leçon | Mesure dans ce carnet |
|---|---|
| L'inatteignabilité se démontre, elle ne se suppose pas | BFS exhaustif dans `A_t` : `36 etats, 36 expansions` — `g1 ... atteignable=False h*=infini`, `g2 ... atteignable=False`, `g3: detenir la cle atteignable=True h*=3` |
| Un échec de recherche n'est pas un certificat : trois budgets, même verdict | `g1` et `g2` échouent aux trois niveaux d'effort : `BFS ECHEC( 36)`, `A*+h_max ECHEC( 36)`, `IDDFS(24) ECHEC( 4151)` ; `g3` passe aux trois (`OK( 36)`, `OK( 4)`, `OK( 24)`) |
| La relaxation ne peut pas inventer la rive droite | `g1: balise (5,2): h_max dans A_t = INFINI (relaxement inatteignable)` — et le carnet cite sa garantie : `le lake garantit h+ <= h* en Admissibility.lean:41` |
| La chaîne d'admissibilité se vérifie sur le but conjonctif | `h* (reel, BFS uniforme) = 8`, `h_max (relaxe, exact) = 3`, `chaine verifiee : h_max (3) <= h* (8) -> STRICT` |
| Le différentiel se mesure : une primitive nommée, trois opérateurs | `A_t : 41 operateurs` ; `A_t+1 : 44 operateurs (primitive ajoutee : ['raft-across-y0', 'raft-across-y1', 'raft-across-y2'])` |
| Ce que la primitive ouvre, en états et en coûts | `\|A_t\| = 36 etats -> \|A_t+1\| = 60 etats (+24)` ; `g1 ECHEC -> OK h*(A_t+1) = 7 NOUVELLEMENT ATTEIGNABLE`, `g2 ECHEC -> OK h* = 6`, `g3 OK -> OK h* = 3` |

## Vérifiez votre compréhension

**Question 1 — Pourquoi le tableau du pas 2 (trois algorithmes) n'ajoute-t-il rien au verdict du pas 1 (BFS exhaustif), et pourquoi le faire quand même ?**

Le pas 1 rend `Ensemble atteignable dans A_t : 36 etats, 36 expansions (BFS exhaustif)` ; le pas 2 rend `Verdict du controle : g1 et g2 echouent aux TROIS niveaux d'effort. L'espace atteignable est le MEME (36 etats) quel que soit le budget.`

<details><summary>Réponse</summary>

Parce que le pas 1 **démontre** (énumération exhaustive des 36 états : rive droite absente), tandis que le pas 2 teste la **robustesse du verdict** face à l'objection « tu n'as pas cherché assez ». IDDFS pousse le budget à `4151` expansions — deux ordres de grandeur au-delà des `36` de l'exhaustif — et ne trouve rien de plus : c'est la signature d'une barrière structurelle (le gouffre sans radeau), pas d'un budget trop court. La mesure asymétrique est la leçon : `g3 OK(4)` en A* contre `OK(36)` en BFS montre aussi ce que le guidage achète quand le but **est** atteignable.

</details>

**Question 2 — Après l'ajout du radeau, h_max(g1) passe de l'infini à 7 : que vérifie la ligne `chaine h_max<=h*: OK` et d'où vient la garantie ?**

Le pas 4 rend `g1: balise (5,2) h_max: A_t=inf -> A_t+1=7 h*(A_t+1)=7 chaine h_max<=h*: OK` et le carnet cite `le lake garantit h+ <= h* en Admissibility.lean:41`.

<details><summary>Réponse</summary>

Elle vérifie numériquement ce que le lake prouve formellement : `h_max <= h+ <= h*` (h_max majore par le fait le plus cher du but relaxé ; h+ est le coût du plan relaxé ; l'admissibilité de la relaxation est le théorème d'`Admissibility.lean:41`, prouvé sans `sorry` dans le companion `Planners-05b-Lean-Relaxation-Lean`). Sur g1 après ajout : `h_max = 7 = h*` — égalité, jamais inversion. Et sur le but conjonctif du contrôle, l'écart était strict (`3 < 8`) : la chaîne autorise l'égalité, elle interdit la surestimation. C'est ce qui rend h_max utilisable pour **certifier** l'inatteignabilité : `h_max = INFINI` dans A_t ne peut pas venir d'une sous-estimation.

</details>

## Pour aller plus loin

- [Index de la sous-série 02-Classical](README.md) — le parcours principal et les paliers d'approfondissement (ce carnet est le palier 5c).
- [Index de la série Planners](../README.md) — la place du protocole de mesure dans la série.

Les carnets voisins sont cités en prose, sans lien. `Planners-05b-Lean-Relaxation-Lean` prouve le théorème (`h+ <= h*`) que ce carnet applique et cite à sa source ; `Planners-05-Heuristics-CSharp` construit les heuristiques (h_max, h+, h_landmark) dont la chaîne est mesurée ici ; `Planners-06-Domains-Python` montre les domaines classiques où ces questions d'atteignabilité se posent à grande échelle. La suite du travail est déjà posée dans le carnet : l'exercice 1 mesure le delta d'une primitive **gratuite** (le deltaplane), l'exercice 2 d'une extension **stérile** (jeter la planchette), l'exercice 3 la chaîne `h_max <= h+ <= h*` sur g3.

> Portage C# de la [version Python](Planners-05c-Differentiel-Atteignabilite-Python.ipynb).